# Step 3B – Modelling Table for the Perfect Cells

**This is a new notebook. It does not replace or overwrite anything from the data-prep notebook or the earlier Step 3.** It only *reads* files from the data-prep output folder and *writes* to a **new folder** with new file names.

**What it does**

| # | Task | Section |
|---|---|---|
| 1 | Keep only the **perfect cells** (no missing hour on normal days) | 2 |
| 2 | Check that no missing rows remain outside the 3 data-gap days | 2 |
| 3 | Hide the traffic value on the 3 data-gap days (Dec 5, 19, 30) | 3 |
| 4 | Build lag features; gaps are filled for the lag **inputs only** | 4 |
| 5 | Add time features: hour of day, day of week, weekend (no holiday features) | 5 |
| 6 | Split by time; the stress test is exactly **21 Dec – 1 Jan** | 6 |
| 7 | Baseline check and save to the new folder | 7–8 |

**Rule to remember:** filled values may be used as *inputs* (lags), but are never used as the *truth* that a model is trained or scored on.

## 1. Mount Google Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ============ EDIT THESE ============
PREP_DIR = '/content/drive/MyDrive/telecom_italia/processed'            # data-prep outputs (only READ from here)
OUT_DIR  = '/content/drive/MyDrive/telecom_italia/step3b_perfect_cells' # NEW folder: everything is saved here

TRAIN_START = '2013-11-08'   # first week is only a warm-up so that 1-week lags exist
TRAIN_END   = '2013-12-14'
VALID_END   = '2013-12-20'   # the surge starts on 21 Dec: everything after this date is the stress test

EXPECTED_PERFECT = 6039      # number of perfect cells found in the data-prep run (used for a sanity check)
OVERWRITE = False            # False = refuse to overwrite an existing output file
# ====================================

os.makedirs(OUT_DIR, exist_ok=True)
print('Reading from :', PREP_DIR)
print('Writing to   :', OUT_DIR)

Reading from : /content/drive/MyDrive/telecom_italia/processed
Writing to   : /content/drive/MyDrive/telecom_italia/step3b_perfect_cells


## 2. Load the data and keep only the perfect cells

`cell_report.csv` (from the data-prep notebook) has, for every cell, `hours_missing_normal` = number of hours missing on **normal days** (all days except the data-gap days). A **perfect cell** has 0.

The table is sorted by cell and time, which is required so that lags are taken within each cell in the right order.

In [ ]:
df = pd.read_parquet(f'{PREP_DIR}/hourly_selected_cells.parquet',
                     columns=['cell_id', 'hour', 'internet', 'is_gap_day', 'is_missing'])
df = df.sort_values(['cell_id', 'hour']).reset_index(drop=True)
print(f'Loaded            : {len(df):,} rows | {df.cell_id.nunique():,} cells (the suitable cells)')

report  = pd.read_csv(f'{PREP_DIR}/cell_report.csv', index_col='cell_id')
perfect = report.index[report['hours_missing_normal'] == 0]
print(f'Perfect cells     : {len(perfect):,}   (expected {EXPECTED_PERFECT:,}) ->',
      'OK' if len(perfect) == EXPECTED_PERFECT else 'DIFFERENT, please check')

df = df[df['cell_id'].isin(perfect)].reset_index(drop=True)
gap_days = sorted(df.loc[df.is_gap_day, 'hour'].dt.normalize().unique())

print(f'After filtering   : {len(df):,} rows | {df.cell_id.nunique():,} cells')
print('Data-gap days     :', [str(d.date()) for d in gap_days])

Loaded            : 9,154,176 rows | 6,152 cells (the suitable cells)
Perfect cells     : 6,039   (expected 6,039) -> OK
After filtering   : 8,986,032 rows | 6,039 cells
Data-gap days     : ['2013-12-05', '2013-12-19', '2013-12-30']


In [ ]:
# CHECK: no missing rows may remain outside the 3 data-gap days
missing_outside_gap = int((df['is_missing'] & ~df['is_gap_day']).sum())
print('Missing rows outside the data-gap days:', missing_outside_gap, '(must be 0)')
assert missing_outside_gap == 0, 'Unexpected missing rows outside the gap days - check the data-prep outputs'

# Each perfect cell must also have the full set of hours
hours_per_cell = df.groupby('cell_id').size()
print('Rows per cell: min', hours_per_cell.min(), '| max', hours_per_cell.max(), '(1488 = 62 days x 24)')

Missing rows outside the data-gap days: 0 (must be 0)
Rows per cell: min 1488 | max 1488 (1488 = 62 days x 24)


## 3. Hide the traffic value on the data-gap days

`target` is the traffic value to predict (the hourly **Internet** value). On the data-gap days it is set to **NaN** for every cell, so it can never be used as truth. The rows and the calendar stay in place; only the value is hidden. This keeps the lags correct.

In [ ]:
df['target'] = df['internet'].where(~df['is_gap_day'])      # NaN on data-gap days

print(f'Rows hidden because of data-gap days : {df.is_gap_day.sum():,}')
print(f'Rows with a usable target so far     : {df.target.notna().sum():,}')

Rows hidden because of data-gap days : 434,808
Rows with a usable target so far     : 8,551,224


## 4. Lag features (gaps filled for the inputs only)

A lag feature says "what was the traffic 24 hours ago / one week ago". Right after a gap day the lag would point at a hidden value. So we first build a helper column `filled`:

1. take `target`
2. if it is NaN, use the **same hour one week earlier**
3. if still NaN, use the **same hour one week later**

`lag_24` and `lag_168` are then built from `filled`. The column `target` itself stays NaN on gap days.

In [ ]:
g = df.groupby('cell_id')['target']
df['filled'] = df['target'].fillna(g.shift(168)).fillna(g.shift(-168))   # same hour 1 week before, then after

gf = df.groupby('cell_id')['filled']
df['lag_24']  = gf.shift(24)     # same hour yesterday
df['lag_168'] = gf.shift(168)    # same hour last week

print('Gap-day rows that received a filled value for the lag inputs:',
      int((df.target.isna() & df.filled.notna()).sum()))

Gap-day rows that received a filled value for the lag inputs: 434808


## 5. Time features

Only features that 62 days can really teach: **hour of day**, **day of week** and **weekend**. There are no holiday features: Christmas appears only once in the data, so a model could memorise it but not learn it. The late-December surge is handled by keeping those days and scoring them separately (section 6).

In [ ]:
df['hour_of_day'] = df['hour'].dt.hour
df['day_of_week'] = df['hour'].dt.dayofweek       # Monday = 0
df['is_weekend']  = df['day_of_week'] >= 5

df[['hour', 'hour_of_day', 'day_of_week', 'is_weekend']].head(3)

,hour,hour_of_day,day_of_week,is_weekend
0,2013-11-01 00:00:00,0,4,False
1,2013-11-01 01:00:00,1,4,False
2,2013-11-01 02:00:00,2,4,False


## 6. Time-based split and the `usable` flag

Data are split **by time, never randomly**:

| split | period | purpose |
|---|---|---|
| warmup | before `TRAIN_START` | only supplies lags |
| train | `TRAIN_START` – `TRAIN_END` | fit models |
| validation | up to `VALID_END` | tune models |
| stress_test | after `VALID_END` (21 Dec – 1 Jan) | late-December surge, reported separately |

A row is **usable** (for training or scoring) only if it has a real target, a 1-week lag, and is not in the warm-up.

In [ ]:
day = df['hour'].dt.normalize()
df['split'] = np.select([day < TRAIN_START, day <= TRAIN_END, day <= VALID_END],
                        ['warmup', 'train', 'validation'], default='stress_test')
df['usable'] = df['target'].notna() & df['lag_168'].notna() & (df['split'] != 'warmup')

summary = df.groupby('split').agg(rows=('usable', 'size'), usable_rows=('usable', 'sum'),
                                  first_day=('hour', 'min'), last_day=('hour', 'max'))
summary['not_usable'] = summary['rows'] - summary['usable_rows']
print('Stress test should start on 2013-12-21:',
      df.loc[df.split == 'stress_test', 'hour'].min().date() == pd.Timestamp('2013-12-21').date())
summary

Stress test should start on 2013-12-21: True


,rows,usable_rows,first_day,last_day,not_usable
split,,,,,
stress_test,1739232,1594296,2013-12-21,2014-01-01 23:00:00,144936
train,5362632,5217696,2013-11-08,2013-12-14 23:00:00,144936
validation,869616,724680,2013-12-15,2013-12-20 23:00:00,144936
warmup,1014552,0,2013-11-01,2013-11-07 23:00:00,1014552


## 7. Quick check: why the stress test is scored separately

A very simple baseline, "same hour last week", is scored on each split. The error is **WAPE** = total absolute error ÷ total actual traffic (lower is better). If the stress test is much worse, one overall score would hide that.

In [ ]:
def wape(y, p):
    return 100 * np.abs(y - p).sum() / np.abs(y).sum()

rows = []
for s in ['train', 'validation', 'stress_test']:
    d = df[(df['split'] == s) & df['usable']]
    rows.append({'split': s, 'usable_rows': len(d),
                 'mean_traffic': round(d['target'].mean(), 1),
                 'WAPE_same_hour_last_week_%': round(wape(d['target'], d['lag_168']), 1)})
pd.DataFrame(rows)

,split,usable_rows,mean_traffic,WAPE_same_hour_last_week_%
0,train,5217696,85.800003,16.500000
1,validation,724680,88.099998,13.800000
2,stress_test,1594296,128.000000,37.599998


## 8. Save to the new folder

| File (in `OUT_DIR`) | Content |
|---|---|
| `modelling_table_perfect_cells.parquet` | all rows after warm-up, with target, lags, time features, `is_gap_day`, `split`, `usable` |
| `perfect_cells_list.csv` | the cell IDs used |

With `OVERWRITE = False` the notebook stops with a message if a file already exists, so nothing is overwritten by accident.

In [ ]:
keep = ['cell_id', 'hour', 'target', 'lag_24', 'lag_168', 'hour_of_day', 'day_of_week',
        'is_weekend', 'is_gap_day', 'split', 'usable']
model_df = df.loc[df['split'] != 'warmup', keep]

out_table = f'{OUT_DIR}/modelling_table_perfect_cells.parquet'
out_cells = f'{OUT_DIR}/perfect_cells_list.csv'
for f in (out_table, out_cells):
    if os.path.exists(f) and not OVERWRITE:
        raise FileExistsError(f'{f} already exists. Change OUT_DIR or set OVERWRITE = True.')

model_df.to_parquet(out_table, index=False)
pd.Series(sorted(perfect), name='cell_id').to_csv(out_cells, index=False)
print(f'Saved {len(model_df):,} rows to {out_table}')
print(sorted(os.listdir(OUT_DIR)))

Saved 7,971,480 rows to /content/drive/MyDrive/telecom_italia/step3b_perfect_cells/modelling_table_perfect_cells.parquet
['modelling_table_perfect_cells.parquet', 'perfect_cells_list.csv']


## How to use this table next
- **Train** on rows with `split == 'train'` and `usable == True` only.
- **Tune** on `validation`, and report `stress_test` separately. A model trained before the surge will probably under-predict it; that is a finding to report, not a bug.
- **Never** score rows where `usable` is False (gap days, missing target, warm-up).
- In the report, state that 3 data-gap days were masked and why.